# PinkMap - Notebook 2: Clean, Explore, and Baseline

**What this does (Weeks 5 and 6 of the class):**
1. Loads the spreadsheet(s) your collection notebook made
2. **Cleans** the data (removes broken or repeated rows)
3. Adds helpful new columns (**features**), like "how much slower than usual?" and "is it rush hour?"
4. **Splits** the data into a training part, a checking part, and a test part
5. Makes **charts** to find patterns
6. Builds a **baseline** guess so the real model later has something to beat

**How to use it:** click **Runtime -> Run all**. Run it again whenever you have collected more data.


## Step 1. Load the data
This looks for your `.csv` files in the PinkMap folder. If you got csv files from teammates, put them in the same folder, or set `UPLOAD_MORE = True` below.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USING_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_FOLDER = "/content/drive/MyDrive/PinkMap"
    USING_DRIVE = True
    print("Google Drive connected.")
except Exception:
    SAVE_FOLDER = "/content/PinkMap"
    print("Google Drive not connected. Using Colab temporary storage instead.")

os.makedirs(SAVE_FOLDER, exist_ok=True)

UPLOAD_MORE = False   # change to True if you want to upload more csv files (for example from teammates)

csv_files = sorted(glob.glob(SAVE_FOLDER + "/*.csv"))

if len(csv_files) == 0 or UPLOAD_MORE:
    print("Please choose your csv file(s) to upload.")
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(SAVE_FOLDER, name), "wb") as f:
            f.write(content)
    csv_files = sorted(glob.glob(SAVE_FOLDER + "/*.csv"))

print("CSV files found:", len(csv_files))
for f in csv_files:
    print("  ", os.path.basename(f))

raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print("\nTotal rows loaded:", len(raw))
raw.head()

## Step 2. Quality check (look before you clean)
How many values are missing? Are there repeated rows?

In [ ]:
print("Missing values in each column:")
print(raw.isna().sum())

print("\nRepeated rows (same trip at the same time):",
      raw.duplicated(subset=["route_id", "collected_at"]).sum())

## Step 3. Clean the data
We remove rows that cannot be trusted, and we write down how many we removed and why. Writing this down is part of the class requirement (data cleaning and quality assessment).

In [ ]:
df = raw.copy()
start_rows = len(df)
log = []

# 1) Turn the time text into a real date and time
df["collected_at"] = pd.to_datetime(df["collected_at"], errors="coerce")

# 2) Make sure number columns are numbers
for col in ["distance_miles", "traffic_seconds", "typical_seconds"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 3) Remove rows missing something important
before = len(df)
df = df.dropna(subset=["collected_at", "distance_miles", "traffic_seconds", "typical_seconds"])
log.append(("Missing time, distance, traffic time, or typical time", before - len(df)))

# 4) Remove rows with impossible values
before = len(df)
df = df[(df["distance_miles"] > 0) & (df["traffic_seconds"] > 0) & (df["typical_seconds"] > 0)]
log.append(("Zero or negative values", before - len(df)))

# 5) Remove repeated rows
before = len(df)
df = df.drop_duplicates(subset=["route_id", "collected_at"])
log.append(("Repeated rows", before - len(df)))

df = df.sort_values("collected_at").reset_index(drop=True)

print("Rows before cleaning:", start_rows)
for reason, count in log:
    print(f"  removed {count:3d}  -  {reason}")
print("Rows after cleaning: ", len(df))

## Step 4. Add helpful new columns (features)
These help the computer see patterns.

| New column | Meaning |
|---|---|
| `traffic_delay_seconds` | traffic time minus typical time (positive = slower than usual) |
| `traffic_status` | slower / faster / normal (more than 30 seconds off counts) |
| `hour`, `day_of_week`, `is_weekend` | when the trip was measured |
| `time_period` | morning, afternoon, evening, or night |
| `rush_hour` | 1 if it is 7-9 AM or 4-6 PM, otherwise 0 |
| `avg_speed_mph` | distance divided by travel time |

In [ ]:
THRESHOLD_SECONDS = 30   # how far from "usual" counts as slower or faster

df["traffic_delay_seconds"] = df["traffic_seconds"] - df["typical_seconds"]

df["traffic_status"] = np.where(df["traffic_delay_seconds"] > THRESHOLD_SECONDS, "slower",
                        np.where(df["traffic_delay_seconds"] < -THRESHOLD_SECONDS, "faster", "normal"))

df["hour"] = df["collected_at"].dt.hour
df["day_of_week"] = df["collected_at"].dt.day_name()
df["is_weekend"] = (df["collected_at"].dt.dayofweek >= 5).astype(int)

df["time_period"] = np.where(df["hour"] < 12, "morning",
                     np.where(df["hour"] < 16, "afternoon",
                     np.where(df["hour"] < 20, "evening", "night")))

df["rush_hour"] = df["hour"].isin([7, 8, 16, 17]).astype(int)

df["avg_speed_mph"] = df["distance_miles"] / (df["traffic_seconds"] / 3600)

df.head()

## Step 5. Split the data: train / validation / test
- **Train (70%)**: the computer learns from these rows
- **Validation (10%)**: used to check and tune while building
- **Test (20%)**: a final exam the model has never seen

We split by **time order** (oldest rows train, newest rows test). That way the model never "peeks" at the future.

In [ ]:
n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.10)

train = df.iloc[:n_train]
val = df.iloc[n_train:n_train + n_val]
test = df.iloc[n_train + n_val:]

print(f"Total {n} rows -> train {len(train)}, validation {len(val)}, test {len(test)}")
if n < 30:
    print("Tip: you have very little data. Collect more runs before trusting any results.")

## Step 6. Describe the data (Week 6)
Basic numbers, like the ones on your presentation slides, but now calculated from real data.

In [ ]:
summary = {
    "Route observations (rows)": len(df),
    "Different trips (routes)": df["route_id"].nunique(),
    "Different collection hours": df["collected_at"].dt.strftime("%Y-%m-%d %H").nunique(),
    "Average distance (miles)": round(df["distance_miles"].mean(), 2),
    "Shortest / longest trip (miles)": f"{df['distance_miles'].min():.2f} / {df['distance_miles'].max():.2f}",
    "Average traffic travel time (minutes)": round(df["traffic_seconds"].mean() / 60, 2),
    "Average size of delay (seconds, ignoring + or -)": round(df["traffic_delay_seconds"].abs().mean(), 1),
    "Correlation: distance vs traffic time": round(df["distance_miles"].corr(df["traffic_seconds"]), 3),
}
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nHow many rows were slower / faster / normal:")
print(df["traffic_status"].value_counts())

## Step 7. Charts: look for patterns

In [ ]:
# Chart 1: traffic time vs typical time (dots above the line = slower than usual)
plt.figure(figsize=(6, 5))
plt.scatter(df["typical_seconds"] / 60, df["traffic_seconds"] / 60, alpha=0.6)
top = max(df["typical_seconds"].max(), df["traffic_seconds"].max()) / 60
plt.plot([0, top], [0, top], linestyle="--", color="gray")
plt.xlabel("Typical travel time (minutes)")
plt.ylabel("Travel time with traffic (minutes)")
plt.title("Traffic time vs typical time\n(above the dashed line = slower than usual)")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: average delay by hour of day
by_hour = df.groupby("hour")["traffic_delay_seconds"].mean()
plt.figure(figsize=(8, 4))
by_hour.plot(kind="bar", color="#b5707f")
plt.axhline(0, color="gray")
plt.xlabel("Hour of day (24-hour clock)")
plt.ylabel("Average delay (seconds)")
plt.title("Is traffic worse at certain hours?")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: average delay for each trip
by_route = df.groupby("route_id")["traffic_delay_seconds"].mean()
plt.figure(figsize=(8, 4))
by_route.plot(kind="bar", color="#1f2f4d")
plt.axhline(0, color="gray")
plt.xlabel("Trip number (route_id)")
plt.ylabel("Average delay (seconds)")
plt.title("Which trips are usually slower than normal?")
plt.tight_layout()
plt.show()

## Step 8. Baseline models
A **baseline** is a simple guess. Later, your real model must beat it.

- **Baseline A:** guess that traffic time = Mapbox's typical time
- **Baseline B:** a small straight-line model that uses distance and typical time

**MAE** = the average mistake in seconds. **RMSE** = like MAE, but big mistakes count extra. Smaller is better.

In [ ]:
from sklearn.linear_model import LinearRegression

def errors(actual, guess):
    e = np.array(actual) - np.array(guess)
    return float(np.mean(np.abs(e))), float(np.sqrt(np.mean(e ** 2)))

target = "traffic_seconds"
inputs = ["distance_miles", "typical_seconds"]

if len(train) < 3 or len(test) < 1:
    print("Not enough data yet. Collect more runs and run this notebook again.")
else:
    # Baseline A
    a_mae, a_rmse = errors(test[target], test["typical_seconds"])

    # Baseline B
    lin = LinearRegression().fit(train[inputs], train[target])
    b_mae, b_rmse = errors(test[target], lin.predict(test[inputs]))

    results = pd.DataFrame({
        "Model": ["A: guess = typical time", "B: linear (distance + typical time)"],
        "Test MAE (sec)": [round(a_mae, 1), round(b_mae, 1)],
        "Test RMSE (sec)": [round(a_rmse, 1), round(b_rmse, 1)],
    })
    print(f"Test rows used: {len(test)}")
    display(results)

## Step 9. Save the cleaned files
These files go in a `processed` folder. The next notebook (the real model) will use them.

In [ ]:
out = os.path.join(SAVE_FOLDER, "processed")
os.makedirs(out, exist_ok=True)

df.to_csv(out + "/clean_data.csv", index=False)
train.to_csv(out + "/train.csv", index=False)
val.to_csv(out + "/val.csv", index=False)
test.to_csv(out + "/test.csv", index=False)
print("Saved clean_data.csv, train.csv, val.csv, test.csv to:", out)

if not USING_DRIVE:
    from google.colab import files
    files.download(out + "/clean_data.csv")

## What to write in your notes for the professor
- **Where the data came from:** OpenStreetMap (places) and the Mapbox Directions API (traffic and typical travel times).
- **What we cleaned:** see the cleaning log in Step 3.
- **What we added:** delay, status, hour, day, rush hour, and speed (Step 4).
- **How we split:** by time, 70/10/20 (Step 5).
- **What we found:** see the charts and numbers in Steps 6-7.
- **Baseline results:** see the table in Step 8. With a small dataset, treat the numbers as a proof of concept, not a final result.
